# Desambiguação e vinculação de entidades

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_10/02_desambiguacao_e_vinculacao.ipynb)

Normalizar reduz variação de escrita; vincular atribui um identificador; desambiguar
decide entre referentes possíveis. Um catálogo de autoridade deve preservar variantes,
fonte da decisão e opção **não resolvido**.

![Menções variantes convergem para um identificador, enquanto homônimos exigem contexto.](imagens/02_vinculacao.svg)

<!-- alinhamento-slides -->
## Sequência lógica deste notebook

Como nos slides, o raciocínio parte de um exemplo ou problema concreto e só
então introduz a técnica. Leia e execute as etapas nesta ordem:

1. **Menção localizada**
2. **Normalização**
3. **Candidatos**
4. **Contexto**
5. **Vínculo ou não resolvido**
6. **Ficha auditável**

A saída de cada etapa fornece a justificativa para a seguinte; o resultado
computacional só se torna evidência depois de interpretado e confrontado com
casos, fontes ou limites.


In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_10'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

## 1. Da forma escrita ao candidato de identidade

Normalizamos variações gráficas para buscar candidatos no catálogo. Em seguida, o contexto confirma, rejeita ou deixa a ligação sem resolução; igualdade de forma, sozinha, não estabelece identidade.


In [ ]:
import pandas as pd,re,unicodedata
docs=pd.read_csv("dados/documentos.csv");aut=pd.read_csv("dados/autoridades.csv").fillna("")
def normalizar(s):
    s=unicodedata.normalize("NFKD",s);return re.sub(r"[^a-z0-9 ]","",s.encode("ascii","ignore").decode().lower()).strip()
indice={}
for _,r in aut.iterrows():
    for f in [r.nome_canonico,*[x for x in r.variantes.split(";") if x]]: indice[normalizar(f)]=r.id_entidade
teste=pd.DataFrame({"mencao":["Ana F.","Dra. Ana Ferreira","Associação Aurora","Santa Rita" ]})
teste["forma_normalizada"]=teste.mencao.map(normalizar);teste["id_proposto"]=teste.forma_normalizada.map(indice)
display(teste)

<!-- complemento-sequencia -->
## 2. Contexto, decisão e ficha auditável

Normalização aproxima grafias; não resolve identidade. Para cada menção, liste
candidatos e use contexto, tipo esperado, data e relações locais para decidir. Quando
a evidência não discrimina candidatos, **não resolvido** é uma saída válida.

| Campo | Exemplo para `Santa Rita` em D008 |
|---|---|
| forma original | Santa Rita |
| candidato automático | L002, lugar |
| contexto | “a embarcação Santa Rita” |
| evidência contrária | o núcleo “embarcação” indica objeto, não lugar |
| decisão | não vincular a L002 |
| confiança | alta para a rejeição; referente permanece não resolvido |

A ficha preserva por que o candidato foi aceito, rejeitado ou mantido em aberto.


“Santa Rita” demonstra por que igualdade textual não basta: em D008 é embarcação; em
outros documentos, lugar. O contexto e o tipo esperado devem poder bloquear a ligação.

### U10-A03 — Atividade — ficha de vinculação

Para quatro menções, registre forma, candidato, contexto, evidência, decisão e grau de
confiança. Inclua D008 como `não vincular a L002`. **Dinâmica (25 minutos):** decisão
individual e adjudicação em trio. **Resposta:** escreva aqui.

## Passagem

Entidades vinculadas permitem representar ações, mas proximidade na frase não prova
relação. A próxima etapa explicita sujeito, predicado, objeto, tempo e evidência.